# MartiniSurf Colab Workflow: Protein Surface Systems

Reproducible pipeline organized in clear stages.

**Recommended flow:**
1. Install environment
2. (Optional) Generate/load CG linker
3. Configure inputs, surface, and orientation
4. Run MartiniSurf
5. Download and visualize


## Step 1. Environment Setup


In [ ]:
#@title Step 1 • Install MartiniSurf

import getpass
import os
import shlex
import shutil
import subprocess
from urllib.parse import quote

REPO_URL = "https://github.com/jjimenezgar/MartiniSurf.git"


def run_cmd(cmd, check=True):
    if isinstance(cmd, str):
        res = subprocess.run(cmd, shell=True, text=True, capture_output=True)
        shown = cmd
    else:
        res = subprocess.run(cmd, text=True, capture_output=True)
        shown = ' '.join(shlex.quote(x) for x in cmd)
    if check and res.returncode != 0:
        raise RuntimeError(f"Command failed ({shown})\nSTDOUT:\n{res.stdout[-4000:]}\nSTDERR:\n{res.stderr[-4000:]}")
    return res


def clone_repo(repo_url: str, dst: str) -> None:
    # Try anonymous clone first.
    res = run_cmd(['git', 'clone', '--depth', '1', repo_url, dst], check=False)
    if res.returncode == 0:
        return

    # If anonymous clone fails, prompt token securely and retry.
    token = getpass.getpass('GitHub token (repo read access): ').strip()
    if not token:
        raise RuntimeError('Clone failed and no token was provided.')

    auth_url = repo_url.replace('https://', f'https://{quote(token, safe="")}@')
    res2 = run_cmd(['git', 'clone', '--depth', '1', auth_url, dst], check=False)
    if res2.returncode != 0:
        raise RuntimeError(
            'Repository clone failed.\n'
            f'Anonymous attempt stderr:\n{res.stderr[-4000:]}\n'
            f'Authenticated attempt stderr:\n{res2.stderr[-4000:]}'
        )


def ensure_python2() -> str | None:
    # 1) Already available in PATH
    py2 = shutil.which('python2.7') or shutil.which('python2')
    if py2:
        os.environ['MARTINISURF_PYTHON2'] = py2
        return py2

    # 2) Try apt-based installation (works on many Colab runtimes)
    run_cmd(['apt-get', 'update'], check=False)
    for pkg in ['python2.7', 'python2', 'python2-minimal']:
        run_cmd(['apt-get', 'install', '-y', pkg], check=False)
        py2 = shutil.which('python2.7') or shutil.which('python2')
        if py2:
            os.environ['MARTINISURF_PYTHON2'] = py2
            return py2

    return None


run_cmd('rm -rf /content/MartiniSurf', check=False)
clone_repo(REPO_URL, '/content/MartiniSurf')

run_cmd(['python', '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
run_cmd(['python', '-m', 'pip', 'install', '-q', '-r', '/content/MartiniSurf/requirements.txt'])
run_cmd(['python', '-m', 'pip', 'install', '-q', '-e', '/content/MartiniSurf'])
run_cmd(['python', '-m', 'pip', 'install', '-q', 'py3Dmol'])

# Protein workflow tools
run_cmd(['python', '-m', 'pip', 'install', '-q', 'martinize2'], check=False)
run_cmd(['python', '-m', 'pip', 'install', '-q', 'vermouth'], check=False)

py2 = ensure_python2()

print('martinisurf:', shutil.which('martinisurf') or 'NOT FOUND')
print('martinize2:', shutil.which('martinize2') or 'NOT FOUND')
print('python2:', py2 or 'NOT FOUND (DNA mode will fail without python2.7)')
print('Installation complete')


## Step 2. Optional Linker Preparation


In [ ]:
#@title Step 2A • Optional Linker Generation (AutoMartini M3)
RUN_LINKER_GENERATION = True  #@param {type:"boolean"}
INPUT_MODE = "SMILES"  #@param ["SMILES", "Upload SMILES file"]
SMILES = "CCCCCCCCC"  #@param {type:"string"}
MOLECULE_NAME = "LNK"  #@param {type:"string"}

import subprocess
from pathlib import Path

# Ensure RDKit is available
subprocess.run(["pip", "install", "-q", "rdkit"], check=False)

GENERATED_LINKER_GRO = None

if RUN_LINKER_GENERATION:

    print("AutoMartini M3 Linker Generation")

    # --- Handle upload mode ---
    if INPUT_MODE == "Upload SMILES file":
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded.")

        filename = next(iter(uploaded.keys()))
        filepath = Path("/content") / filename

        content = filepath.read_text().strip().split()
        if not content:
            raise RuntimeError("Uploaded file is empty.")

        SMILES = content[0]
        print(f"Loaded SMILES from file: {SMILES}")

    if not SMILES.strip():
        raise RuntimeError("SMILES string is empty.")

    if not MOLECULE_NAME.strip():
        raise RuntimeError("Molecule name is empty.")

    # --- Minimal deps ---
    subprocess.run(
        ["pip", "install", "-q", "numpy", "scipy", "networkx"],
        check=True
    )

    # --- Clone repo if needed ---
    if not Path("/content/Automartini_M3").exists():
        subprocess.run(
            [
                "git", "clone",
                "https://github.com/Martini-Force-Field-Initiative/Automartini_M3"
            ],
            check=True
        )

    # --- Install AutoMartini as editable package ---
    subprocess.run(
        ["pip", "install", "-e", "/content/Automartini_M3"],
        check=True
    )

    # --- Execute AutoMartini ---
    cmd = [
        "python",
        "-m",
        "auto_martiniM3",
        "--mode", "run",
        "--smi", SMILES,
        "--mol", MOLECULE_NAME
    ]

    print(f"Running AutoMartini M3 for '{MOLECULE_NAME}' ...")
    run = subprocess.run(cmd, capture_output=True, text=True)

    if run.returncode != 0:
        print(run.stderr[-4000:])
        raise RuntimeError("AutoMartini M3 failed.")

    # --- Check output ---
    itp_file = Path(f"/content/{MOLECULE_NAME}.itp")
    gro_file = Path(f"/content/{MOLECULE_NAME}.gro")

    if itp_file.exists() and gro_file.exists():
        GENERATED_LINKER_GRO = str(gro_file)
        print("Linker successfully generated")
        print(f" - {itp_file.name}")
        print(f" - {gro_file.name}")
    else:
        print("Output files not found in /content")
        subprocess.run(["ls", "-lh", "/content"])

else:
    print("Linker generation skipped.")


In [ ]:
#@title Step 2B • Load Optional Coarse-Grained Linker Files
#@markdown ### Linker Files
#@markdown Upload `linker.gro` and `linker.itp` (matching basename preferred).

from pathlib import Path
import shutil
from google.colab import files

LINKER_PATH = None

print('Optional: upload linker.gro and linker.itp (matching basename preferred).')
uploaded = files.upload()

if uploaded:
    names = list(uploaded.keys())
    gros = [n for n in names if n.lower().endswith('.gro')]
    itps = [n for n in names if n.lower().endswith('.itp')]

    if not gros:
        raise RuntimeError('Please upload one linker .gro file')
    if not itps:
        raise RuntimeError('Please upload one linker .itp file')

    linker_gro = Path('/content') / gros[0]
    linker_itp = Path('/content') / itps[0]
    target_itp = linker_gro.with_suffix('.itp')
    if linker_itp != target_itp:
        shutil.copyfile(linker_itp, target_itp)

    LINKER_PATH = str(linker_gro)

if LINKER_PATH is None and GENERATED_LINKER_GRO is not None:
    LINKER_PATH = GENERATED_LINKER_GRO

print('LINKER_PATH:', LINKER_PATH)


## Step 3. Build Configuration


In [ ]:
#@title Step 3A • Inputs and Force-Field Controls
#@markdown ### Core Inputs
#@markdown **PDB ID**: use a 4-character ID (for example `1RJW`) unless uploading a file.
#@markdown **Upload PDB File**: enable to upload a local structure file.
PDB_ID = "1RJW" #@param {type:"string"}
UPLOAD_PDB_FILE = False #@param {type:"boolean"}

MOLTYPE = "Protein" #@param {type:"string"}
USE_DSSP = True #@param {type:"boolean"}
USE_GO_MODEL = False #@param {type:"boolean"}
GO_EPS = 9.414 #@param {type:"number"}
USE_ELASTIC = False #@param {type:"boolean"}
ELASTIC_EF = 700 #@param {type:"integer"}
MARTINIZE_MAXWARN = 1 #@param {type:"integer"}
MERGE_GROUPS = "A,B,C,D" #@param {type:"string"}

PDB_INPUT_VALUE = PDB_ID.strip()

if UPLOAD_PDB_FILE:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No PDB uploaded')
    PDB_INPUT_VALUE = '/content/' + next(iter(uploaded.keys()))

print('PDB input:', PDB_INPUT_VALUE)


In [ ]:
#@title Step 3B • Surface Setup
#@markdown ### Surface Setup
#@markdown **Upload Surface File**: enable to upload `surface.gro` directly.
#@markdown If disabled, a new surface is generated from topology, edge size, and bond length settings.
UPLOAD_SURFACE_FILE = False #@param {type:"boolean"}
SURFACE_TOPOLOGY = "4-1" #@param ["4-1", "2-1"]
BOND_LENGHT_NM = 0.47 #@param {type:"number"}
EDGE_LENGHT_X_NM = 20.0 #@param {type:"number"}
EDGE_LENGHT_Y_NM = 20.0 #@param {type:"number"}
SURFACE_BEAD = "P4" #@param {type:"string"}

SURFACE_PATH = None
DX_NM = None

if UPLOAD_SURFACE_FILE:
    from google.colab import files
    print('Upload surface.gro file')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No surface file uploaded')
    SURFACE_PATH = '/content/' + next(iter(uploaded.keys()))
else:
    DX_NM = float(BOND_LENGHT_NM)
    if DX_NM <= 0:
        raise ValueError('BOND_LENGHT_NM must be > 0.')
    print(f'Generated surface preset: {SURFACE_TOPOLOGY} (dx={DX_NM} nm)')


In [ ]:
#@title Step 3C • Orientation Setup
#@markdown ### Orientation
#@markdown **Residue Selection**: group format `GROUP RESID [RESID ...]`, separated by semicolons.
#@markdown The same residue selection field is used for Anchor mode and Linker mode.
ORIENTATION_MODE = "Anchor mode" #@param ["Anchor mode", "Linker mode"]
RESIDUE_SELECTION = "1 8 10 11; 2 1025 1027 1028" #@param {type:"string"}
TARGET_DISTANCE_A = 10.0 #@param {type:"number"}
INVERT_LINKER = False #@param {type:"boolean"}


def parse_group_lines(raw_text, field_name):
    groups = []
    for ln in (raw_text or '').replace(';', '\n').splitlines():
        line = ln.strip()
        if not line:
            continue
        parts = line.replace(',', ' ').split()
        if len(parts) < 2:
            raise ValueError(f'{field_name}: invalid line "{line}". Use: GROUP RESID [RESID ...]')
        groups.append(parts)
    return groups

print('Orientation mode:', ORIENTATION_MODE)


In [ ]:
#@title Step 3D • Configure Optional Surface Linkers
#@markdown ### Extra Surface Linkers
#@markdown Set the number of additional random linkers placed on the surface (default `0`).
SURFACE_LINKERS = 0 #@param {type:"integer"}
print('Surface linkers:', int(SURFACE_LINKERS))


In [ ]:
#@title Step 3E • Run MartiniSurf
import importlib.util
import shlex
import shutil
import subprocess


def parse_merge_groups(raw_text):
    vals = []
    for token in (raw_text or '').replace(';', '\n').splitlines():
        item = token.strip()
        if item:
            vals.append(item)
    return vals

if shutil.which('martinisurf') is None:
    raise RuntimeError('martinisurf is not installed. Run Step 1 first.')
if shutil.which('martinize2') is None:
    raise RuntimeError('martinize2 is not installed. Run Step 1 first.')

if USE_DSSP:
    has_mdtraj = importlib.util.find_spec('mdtraj') is not None
    has_mkdssp = shutil.which('mkdssp') is not None
    if not has_mdtraj and not has_mkdssp:
        print('Warning: USE_DSSP=True but mdtraj/mkdssp are not detected in PATH.')

# Robust fallback in case Step 2B was skipped.
LINKER_PATH = globals().get('LINKER_PATH', None)
if LINKER_PATH is None:
    LINKER_PATH = globals().get('GENERATED_LINKER_GRO', None)

SURFACE_LINKERS_VALUE = int(globals().get('SURFACE_LINKERS', 0))

OUTDIR = 'Simulation_Files'
cmd = [
    'martinisurf',
    '--pdb', PDB_INPUT_VALUE,
    '--moltype', MOLTYPE,
    '--outdir', OUTDIR,
    '--maxwarn', str(int(MARTINIZE_MAXWARN)),
]

if USE_GO_MODEL:
    cmd += ['--go']
    if str(GO_EPS).strip():
        cmd += ['--go-eps', str(GO_EPS).strip()]
if not USE_DSSP:
    cmd += ['--no-dssp']
if USE_ELASTIC:
    cmd += ['--elastic', '--ef', str(int(ELASTIC_EF))]

for mg in parse_merge_groups(MERGE_GROUPS):
    cmd += ['--merge', mg]

if UPLOAD_SURFACE_FILE:
    if not SURFACE_PATH:
        raise RuntimeError('UPLOAD_SURFACE_FILE is enabled but no surface file was uploaded in Step 3B.')
    cmd += ['--surface', SURFACE_PATH]
else:
    cmd += [
        '--surface-mode', SURFACE_TOPOLOGY,
        '--lx', str(EDGE_LENGHT_X_NM),
        '--ly', str(EDGE_LENGHT_Y_NM),
        '--dx', str(DX_NM),
        '--surface-bead', SURFACE_BEAD,
    ]

if ORIENTATION_MODE == 'Linker mode':
    if LINKER_PATH is None:
        raise RuntimeError('Linker mode selected but no linker is available. Use Step 2A or Step 2B.')
    cmd += ['--linker', LINKER_PATH]
    if INVERT_LINKER:
        cmd += ['--invert-linker']
    if SURFACE_LINKERS_VALUE > 0:
        cmd += ['--surface-linkers', str(SURFACE_LINKERS_VALUE)]

    groups = parse_group_lines(RESIDUE_SELECTION, 'RESIDUE_SELECTION')
    if not groups:
        raise ValueError('RESIDUE_SELECTION is empty. Add at least one group line.')
    for g in groups:
        cmd += ['--linker-group'] + g
else:
    groups = parse_group_lines(RESIDUE_SELECTION, 'RESIDUE_SELECTION')
    if not groups:
        raise ValueError('RESIDUE_SELECTION is empty. Add at least one group line.')
    for g in groups:
        cmd += ['--anchor'] + g
    cmd += ['--dist', str(TARGET_DISTANCE_A)]

print('Running:\n' + ' '.join(shlex.quote(x) for x in cmd))
res = subprocess.run(cmd, text=True, capture_output=True)
print(res.stdout[-12000:])
if res.returncode != 0:
    print(res.stderr[-12000:])
    combined = ((res.stdout or '') + '\n' + (res.stderr or '')).lower()
    msg = 'MartiniSurf failed. '
    if 'warnings were encountered after accounting for the -maxwarn flag' in combined:
        msg += 'Increase MARTINIZE_MAXWARN (for example 1 -> 3).'
    elif 'dssp' in combined and ('failed' in combined or 'not found' in combined):
        msg += 'Likely cause: DSSP setup issue.'
    elif 'martinize2' in combined and ('not found' in combined or 'no such file' in combined):
        msg += 'Likely cause: martinize2 missing in PATH.'
    elif 'failed to download rcsb' in combined:
        msg += 'Likely cause: network issue or invalid PDB ID.'
    raise RuntimeError(msg)

print('Build completed')


## Step 4. Visual Quality Check


In [ ]:
#@title Step 4 • Viewer
#@markdown ### Visualization Options
#@markdown **Style** controls global rendering. **Highlight Residues** colors selected residue IDs in red.
STYLE = "Spheres" #@param ["Spheres", "Sticks"]
HIGHLIGHT_RESIDUES = "8 ,10 ,11,1025 ,1027 ,1028" #@param {type:"string"}

import py3Dmol
from pathlib import Path

SYSTEM_GRO_FILE = "Simulation_Files/2_system/immobilized_system.gro"
path = Path('/content') / SYSTEM_GRO_FILE
if not path.exists():
    raise FileNotFoundError(f'File not found: {path}')

model = path.read_text()
view = py3Dmol.view(width='100%', height=800)
view.addModel(model, 'gro')
if STYLE == 'Sticks':
    view.setStyle({'stick': {}})
else:
    view.setStyle({'sphere': {'radius': 1.4}})

res_tokens = [x.strip() for x in HIGHLIGHT_RESIDUES.replace(';', ',').split(',') if x.strip()]
if res_tokens:
    resi = [int(x) for x in res_tokens]
    if STYLE == 'Sticks':
        view.addStyle({'resi': resi}, {'stick': {'color': 'red'}})
    else:
        view.addStyle({'resi': resi}, {'sphere': {'color': 'red', 'radius': 1.7}})

view.zoomTo()
view.show()


## Step 5. Export Results


In [ ]:
#@title Step 5 • Download Results

import shutil
from pathlib import Path
from google.colab import files

OUTPUT_FOLDER = "Simulation_Files"
ZIP_NAME = "Simulation_Files"

folder = Path('/content') / OUTPUT_FOLDER
if not folder.exists():
    raise FileNotFoundError(f'Folder not found: {folder}')

shutil.make_archive(ZIP_NAME, 'zip', str(folder))
files.download(ZIP_NAME + '.zip')


## Acknowledgements, Licenses, and Citations
This notebook uses external tools and libraries with their own licenses and citation requirements.

### Please cite
- **Martinize2 / Vermouth**: Kroon PC, Grünewald F, Barnoud J, et al. *Martinize2 and Vermouth: Unified Framework for Topology Generation*. eLife (reviewed preprint v2). DOI: `10.7554/eLife.90627.2`.
- **AutoMartini M3**: Szczuka M, Pereira GP, Walter LJ, Gueroult M, Poulain P, Bereau T, Souza PCT, Chavent M. *Fast Parametrization of Martini3 Models for Fragments and Small Molecules*. Journal of Chemical Theory and Computation, 2025, 22(1):610-623.

### Licenses
- **MartiniSurf**: this repository license.
- **martinize2 / Vermouth**: see upstream project license.
- **AutoMartini M3**: see upstream project license.
- Python packages used here (depending on workflow): `numpy`, `scipy`, `MDAnalysis`, `mdtraj`, `py3Dmol`.
